# Imports

In [ ]:
import sys
import pandas as pd
import os
import numpy as np
import plotly.io as pio
import plotly.express as px
import seaborn as sns
sys.path.append('..')
from plotly.graph_objs import Figure
from functools import reduce
from definitions import ROOT_DIR, DATA_DIR

# Utils

In [2]:
FIGURES_DIR = os.path.join(ROOT_DIR, 'assets', 'corpus_statistics')
os.makedirs(FIGURES_DIR, exist_ok=True)

In [3]:
pio.templates.default = "seaborn"

In [4]:
def save_fig(fig : Figure, filename : str) -> None:
    fig.write_image(os.path.join(FIGURES_DIR, filename))

def show(fig : Figure) -> None:
    fig.update_layout(font_size = 12)
    fig.show()

# Corpus

- Load the data

In [6]:
config = 'ls2n'
data_path = os.path.join(DATA_DIR, config, 'annotate', 'data.jsonl')
data = pd.read_json(data_path, lines=True)

In [7]:
data.head()

,id,title,abstract,keywords,authors,venues,teams,date,labels
0,5420658,Machine Learning Based Environmental Impact Pr...,The construction sector significantly contribu...,"[machine learning, environment product declara...","[Bishwash Neupane, Benoit Hilloulin, Raphael C...","[spi.gciv.ec, info.info-ia, info.info-lg, sde.ie]","[LS2N, LS2N-OGRE, LS2N-CPS3]",2026-11-21 00:00:00+00:00,"[Eco-design, Industrial Digitalization]"
1,5514252,DSFR-FL: Detecting Selfish Free-Riders in Fede...,The digital transformation toward Industry 4.0...,"[federated learning fl, free riders, intrusion...","[Houssem Jmal, Kandaraj Piamrat, Yusheng Ji]",[info.info-ai],"[LS2N, LS2N-STACK]",2026-05-18 00:00:00+00:00,"[Monitoring and Analysis, Industrial Digitaliz..."
2,5495073,Towards a Framework of Open-Source Hardware an...,Human Activity Recognition ( HAR ) consists of...,"[computer vision, recurrent neural network, in...",[Jérôme Rocheteau],[info],"[LS2N, LS2N-VELO]",2026-05-09 00:00:00+00:00,"[Monitoring and Analysis, Industrial Digitaliz..."
3,5266378,A Scaled Poisson Bayesian Model for Viral Epid...,Monitoring an ongoing epidemic requires accura...,"[epidemiology, scaled likelihood bayesian mode...","[Barbara Pascal, Patrice Abry, Juliette Cheval...","[phys.phys.phys-data-an, spi.signal]","[LS2N, LS2N-SIMS]",2026-05-02 00:00:00+00:00,[Monitoring and Analysis]
4,5512410,SCRAPL: Scattering Transform with Random Paths...,The Euclidean distance between wavelet scatter...,"[scattering transform, ddsp, perceptual qualit...","[Christopher Mitcheltree, Vincent Lostanlen, E...","[info.info-sd, info.info-lg, info.info-ts]","[LS2N, LS2N-SIMS]",2026-04-23 00:00:00+00:00,[]


- By Year

In [8]:
date = data['date'].apply(lambda x : int(str(x)[:4])).value_counts()
date.index <= 2015

array([False, False, False, False, False,  True, False,  True,  True,
        True, False, False,  True,  True, False,  True, False,  True,
        True,  True,  True,  True,  True, False,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True])

In [9]:
date = data['date'].apply(lambda x : int(str(x)[:4])).value_counts()
v = date[date.index <= 2015].sum()
date = date[date.index > 2015]
date.index = date.index.map(str)
date.loc['<= 2015'] = v
date = date.iloc[date.index.map(lambda x : str(x)[-4:]).argsort()].reset_index()
date

,date,count
0,<= 2015,3618
1,2016,304
2,2017,371
3,2018,412
4,2019,350
5,2020,375
6,2021,416
7,2022,453
8,2023,445
9,2024,476


In [10]:
fig = px.bar(
    data_frame=date,
    x="date",
    y="count",
    text = "count",
    width=800,
    height=550
)
save_fig(fig, 'year_distribution.svg')
show(fig)

- Keywords

In [11]:
keywords = reduce(lambda a,b : a + b, data['keywords'])
keywords = list(map(str.lower, keywords))
index, count = np.unique(keywords,return_counts=True)
keywords = pd.Series(index=index,data=count).sort_values(ascending=False)
print(len(keywords))

11798


In [12]:
fig = px.bar(
    data_frame = keywords.iloc[:13].sort_values(ascending=True).reset_index(),
    y = "index",
    x = 0,
    text = 0,
    width=800,
    height=450
)
fig.update_layout(xaxis_title="Frequency",yaxis_title="Keyword")
save_fig(fig, 'top_10_most_frequent_keywords.svg')
show(fig)

- Categories

In [13]:
sub_categories = reduce(lambda a,b : a + b, data['venues'])
sub_categories = pd.Series(sub_categories).value_counts()
categories = sub_categories.groupby(by=lambda x : x.split('.')[0]).sum().sort_values(ascending=False)
categories

info    7833
spi     2531
sdv      598
math     399
phys     313
shs      226
scco     205
stat     160
sde       95
sdu       43
nlin      31
chim      18
qfin       7
Name: count, dtype: int64

In [14]:
fig = px.bar(
    data_frame = categories.sort_values(ascending=True).reset_index(),
    y = "index",
    x = "count",
    text = "count",
    width=800,
    height=450
)
fig.update_layout(xaxis_title="Frequency",yaxis_title="Category")
save_fig(fig, 'top_10_most_frequent_categories.svg')
show(fig)

- Teams

In [18]:
teams = list(set(reduce(lambda a,b : a + b, data["teams"])))
teams.remove('LS2N')

for team in teams:
    data[team] = data["teams"].apply(lambda x : team in x)

In [19]:
fig = px.bar(
    data_frame = data[teams]
        .sum()
        .reset_index()
        .rename(columns = { "index" : "Team", 0 : "Number Of Articles" })
        .sort_values(by = "Number Of Articles"),
    y = "Team",
    x = "Number Of Articles",
    text = "Number Of Articles",
    width = 960,
    height = 600
)

save_fig(fig, 'teams_distribution.svg')
fig.show()

- DSIs

In [20]:
data['labels'].apply(lambda x : len(x) > 0).value_counts()

labels
True     4510
False    3211
Name: count, dtype: int64

In [21]:
# classified Vs Not classified
px.pie(
    names = ['Labeled', 'Not labeled'],
    values = data['labels'].apply(lambda x : len(x) > 0).value_counts(),
    width = 850,
    height = 600
)

In [22]:
# Number of Labels
labels = reduce(lambda a, b : a+b, data['labels'].tolist())
labels = pd.Series(labels).value_counts()
labels.reset_index()

,index,count
0,Industrial Digitalization,2238
1,Digital Engineering of Products and Processes,1582
2,Industrial Organization Performance,1360
3,Robotics,1340
4,Monitoring and Analysis,1058
5,Processes and Assembly,525
6,Augmentation of Human Capabilities in Manufact...,476
7,User-Centered Design,386
8,Decarbonization of Energy,134
9,Decarbonization of Mobility,100


In [23]:
fig = px.bar(
    data_frame = labels
        .sort_values()
        .reset_index()
        .rename(columns={ 'index' : 'DSI', 'count' : 'Number Of Articles' }),
    y = "DSI",
    x = "Number Of Articles",
    text = "Number Of Articles",
    width = 960,
    height = 600
)

save_fig(fig, 'dsis_distribution.svg')
fig.show()

- Team By Domain :

In [28]:
teams_labels = []

for idx, row in data.iterrows():

    for team in row['teams']:

        if team == 'LS2N':
            continue

        for label in row['labels']:
            teams_labels.append({ "team" : team, "label" : label })

teams_labels = pd.DataFrame(teams_labels)
teams_labels = teams_labels[['team', 'label']].value_counts().reset_index()

In [29]:
fig = px.bar(
    data_frame = teams_labels,
    x = "count",
    y = "team",
    color = "label",
    width = 1200,
    height = 600,
)
fig.update_layout(barmode='stack', yaxis={'categoryorder':'total ascending'})
fig.update_layout(xaxis_title = "Number of Articles", yaxis_title = "Team")
save_fig(fig, 'teams_by_sub_domain.svg')
show(fig)

In [30]:
fig = px.bar(
    data_frame = teams_labels,
    x = "count",
    y = "label",
    color = "team",
    width = 1200,
    height = 600
)
fig.update_layout(barmode='stack', yaxis={'categoryorder':'total ascending'})
fig.update_layout(xaxis_title = "Number of Articles", yaxis_title = "Sub-Domain")
save_fig(fig, 'sub_domains_by_team.svg')
show(fig)